The idea is to generate a matrix where the rows and columns are transcription factors. The "score" column indicates the score of the binding of the transcription factor in the row being analyzed to the promoter of another transcription factor, which is the one in the column being analyzed.

Each TSV file should be processed as follows:

1. Remove duplicates and save the first sequence_name that appears.
2. The sequence_name is the TF (transcription factor) to whose promoter the TF (that names the TSV file) is binding.
3. The target should be identified.

The generation of the second matrix, the score matrix, follows the same strategy as the p-value matrix, but instead of storing the p-value, it stores the binding score.

In [3]:
# Import libraries
import os
import pandas as pd

In [4]:
# Define the directory where the .narrowPeak files are located
directory = "C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Matriz_score_pvalue/"
matriz_path = "Matrix_File.csv"
matriz = pd.read_csv(matriz_path, sep=";")

# Remove rows that contain NaN in the 'FT' column
matriz = matriz.dropna(subset=[matriz.columns[0]])

# Convert the first column to string and remove whitespace
matriz.iloc[:, 0] = matriz.iloc[:, 0].astype(float).astype(int).astype(str).str.strip()

# Get the list of files in the directory that end in .tsv
narrowpeak_files = [f for f in os.listdir(directory) if f.endswith(".tsv")]

# Read each TSV file
for filename in narrowpeak_files:
    file_path = os.path.join(directory, filename)
    
    # Check if the file is empty
    if os.stat(file_path).st_size == 0:
        print(f"The file {filename} is empty. Skipping.")
        continue
    
    try:
        # Read the TSV file
        file = pd.read_csv(file_path, sep="\t")
    except pd.errors.EmptyDataError:
        print(f"The file {filename} does not contain valid data. Skipping.")
        continue
    
    file = file[["sequence_name", "p-value", "score"]]

    # Remove rows where 'sequence_name' is NaN
    file = file.dropna(subset=["sequence_name"])

    # Convert 'sequence_name' to string
    file["sequence_name"] = file["sequence_name"].astype(int)
    file["sequence_name"] = file["sequence_name"].astype(str)

    # Filter rows by removing duplicates in the 'sequence_name' column
    file = file.drop_duplicates(subset="sequence_name", keep='first')
    
    # Extract the ID from the filename
    id_tsv = str(filename.split("_")[0].strip())  # Ensure that id_tsv is a string
    file = file[["sequence_name", "p-value", "score"]]  # Remove rows where 'sequence_name' is NaN
    file = file.dropna(subset=["sequence_name"])

    # Convert 'sequence_name' to string
    file["sequence_name"] = file["sequence_name"].astype(int)
    file["sequence_name"] = file["sequence_name"].astype(str)

    # Check if the ID is in the first column of the matrix
    if id_tsv in matriz.iloc[:, 0].values:
       # print(f"ID {id_tsv} found in the matrix.")
        
        # Get the index of the row where the ID is found
        row_index = matriz.index[matriz.iloc[:, 0] == id_tsv].tolist()[0]

        # Iterate through the matrix columns
        for col in matriz.columns[1:]:  # Ignore the first column that contains the IDs
            # Convert the column name to string for comparison
            col_name = str(col).strip()
           # print(f"Current column: {col_name}")
            
            # Iterate over the 'sequence_name' values in the TSV file
            for seq_name in file['sequence_name'].values:
                # Convert seq_name to string and remove whitespace
                seq_name = str(seq_name).strip()
                
                # Print the type and value of seq_name for debugging
              # print(f"Type of seq_name: {type(seq_name)}, Value of seq_name: '{seq_name}'")
                
                # Compare the column name with the sequence name
                if col_name == seq_name:
                    # Get the corresponding p-value
                    p_value = file[file['sequence_name'] == seq_name]['p-value'].values[0]
                    
                    # Update the matrix in the corresponding row and column
                    matriz.at[row_index, col] = p_value
                   # print(f"Updated matrix at row {row_index} and column {col} with p-value {p_value}.")
    else:
        print(f"ID {id_tsv} not found in the matrix.")

# Print the complete updated matrix to verify the final results
print("Complete updated matrix:")
print(matriz)

# Save the updated matrix in a new CSV file
matriz_actualizada_path = "Updated_Matrix_pvalue.txt"
matriz.to_csv(matriz_actualizada_path, sep="\t", index=False)
print(f"Updated matrix saved in {matriz_actualizada_path}.")


C:\Users\34698\AppData\Local\Temp\ipykernel_20768\27539636.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0     1892227
1     1896456
2     1909793
3     1923388
4     1964875
5     1874682
6     1877883
7     1878704
8     1887884
9     1888417
10    1894523
11    1897841
12    1898351
13    1906309
14    1914728
15    1917971
16    1919256
17    1873322
18    1873386
19    1879465
20    1882186
21    1889855
22    1891435
23    1894449
24    1896330
25    1898630
26    1906134
27    1906563
28    1908146
29    1910058
30    1915426
31    1923830
32    1958195
33    1762580
34    1884218
35    1886399
36    1890979
37    1891104
38    1909902
39    1980288
40    1880317
41    1882389
42    1882801
43    1887601
44    1894049
45    1916957
46    1918061
47    1921537
48    1950641
49    1799646
50    1888608
51    1891228
52    1902385
53    1789727
54    1954201
55    1867537
56    1917213
57    1877576
Name

Complete updated matrix:
         FT   1892227       1896456   1909793   1923388       1964875  \
0   1892227       NaN  1.110000e-04       NaN  0.000266           NaN   
1   1896456  0.000463  3.880000e-04       NaN  0.000064           NaN   
2   1909793       NaN           NaN  0.000685  0.000242           NaN   
3   1923388  0.000537  4.170000e-04       NaN  0.000069           NaN   
4   1964875  0.000457  5.160000e-04       NaN  0.000026  9.440000e-04   
5   1874682  0.000274  2.420000e-04  0.000027  0.000120  4.390000e-07   
6   1877883  0.000098  3.120000e-06  0.000002  0.000259  6.860000e-11   
7   1878704       NaN  9.400000e-05  0.000877  0.000809  2.030000e-04   
8   1887884       NaN  1.900000e-04  0.000598       NaN  3.390000e-04   
9   1888417  0.000091  3.780000e-05  0.000009  0.000003  6.430000e-07   
10  1894523  0.000124           NaN  0.000179  0.000012  2.530000e-04   
11  1897841       NaN  3.960000e-05  0.000799  0.000657  1.660000e-04   
12  1898351  0.000351  2.7

In [7]:
# Define the directory where the TF files are located
directory = "C:/Users/34698/OneDrive/Escritorio/Trabajos_2024/Factores de transcripcion/DAP/Matriz_score_pvalue/"
matriz_path = "Matrix_File.csv"
matriz = pd.read_csv(matriz_path, sep=";")

# Remove rows that contain NaN in the FT column
matriz = matriz.dropna(subset=[matriz.columns[0]])

# Convert the first column to string and remove white spaces
matriz.iloc[:, 0] = matriz.iloc[:, 0].astype(float).astype(int).astype(str).str.strip()

# Get the list of files in the directory that end in .tsv
narrowpeak_files = [f for f in os.listdir(directory) if f.endswith(".tsv")]

# Read each TSV file
for filename in narrowpeak_files:
    file_path = os.path.join(directory, filename)
    
    # Check if the file is empty
    if os.stat(file_path).st_size == 0:
        print(f"The file {filename} is empty. Skipping.")
        continue
    try:
        # Read the TSV file
        file = pd.read_csv(file_path, sep="\t")
    except pd.errors.EmptyDataError:
        print(f"The file {filename} does not contain valid data. Skipping.")
        continue
    file = file[["sequence_name", "p-value", "score"]]
    
    # Remove rows where sequence name is NaN 
    file = file.dropna(subset=["sequence_name"])
    
    # Convert sequence name to str, first convert it to int and then to str
    file["sequence_name"] = file["sequence_name"].astype(int)
    file["sequence_name"] = file["sequence_name"].astype(str)
    
    # Filter the rows of each TSV file by removing duplicates and keeping the first
    file = file.drop_duplicates(subset="sequence_name", keep='first')
    
    # Extract the ID from the filename
    id_tsv = str(filename.split("_")[0].strip())
    
    # Check if the ID is in the first column of the matrix 
    if id_tsv in matriz.iloc[:, 0].values:
        
        # Get the index of the row where the ID is found
        row_index = matriz.index[matriz.iloc[:, 0] == id_tsv].tolist()[0]
        
        # Iterate through the matrix columns
        for col in matriz.columns[1:]:  # Ignore the first column because it contains the IDs
            col_name = str(col).strip()
            # Iterate over the sequence_name values in the TSV file
            for seq_name in file["sequence_name"].values:
                
                # Convert seq_name to string and remove white spaces
                seq_name = str(seq_name).strip()
                
                # Compare the column name with the sequence name
                if col_name == seq_name:
                    # Get the corresponding score value:
                    score = file[file["sequence_name"] == seq_name]["score"].values[0]
                    
                    # Update the matrix in the corresponding row and column:
                    matriz.at[row_index, col] = score
                    
    else: 
        print(f"ID {id_tsv} not found in the matrix.")

# Print the complete updated matrix to verify the final results
print("Complete updated matrix:")
print(matriz)

# Save the updated matrix in a new CSV file
matriz_actualizada_path = "Updated_Matrix_score.txt"
matriz.to_csv(matriz_actualizada_path, sep="\t", index=False)
print(f"Updated matrix saved in {matriz_actualizada_path}.")


C:\Users\34698\AppData\Local\Temp\ipykernel_20768\2849405348.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0     1892227
1     1896456
2     1909793
3     1923388
4     1964875
5     1874682
6     1877883
7     1878704
8     1887884
9     1888417
10    1894523
11    1897841
12    1898351
13    1906309
14    1914728
15    1917971
16    1919256
17    1873322
18    1873386
19    1879465
20    1882186
21    1889855
22    1891435
23    1894449
24    1896330
25    1898630
26    1906134
27    1906563
28    1908146
29    1910058
30    1915426
31    1923830
32    1958195
33    1762580
34    1884218
35    1886399
36    1890979
37    1891104
38    1909902
39    1980288
40    1880317
41    1882389
42    1882801
43    1887601
44    1894049
45    1916957
46    1918061
47    1921537
48    1950641
49    1799646
50    1888608
51    1891228
52    1902385
53    1789727
54    1954201
55    1867537
56    1917213
57    1877576
Na

Complete updated matrix:
         FT   1892227    1896456   1909793   1923388   1964875    1874682  \
0   1892227       NaN  10.006100       NaN   8.49091       NaN   9.206060   
1   1896456   7.43636   7.775760       NaN  11.01820       NaN   8.763640   
2   1909793       NaN        NaN   7.13855   8.97590       NaN   7.692770   
3   1923388   7.09091   7.642420       NaN  10.87880       NaN   9.218180   
4   1964875   7.77108   7.548190       NaN  12.28310   6.43976   7.228920   
5   1874682   8.33526   8.560690  12.17340   9.80925  17.27750  13.601200   
6   1877883   9.29714  15.108600  15.78290   7.34857  25.46860  15.697100   
7   1878704       NaN  10.781800   7.54545   7.75152   9.86667  11.121200   
8   1887884       NaN   9.818180   8.49091       NaN   9.27879   8.939390   
9   1888417  10.47700  11.655200  13.39080  14.54020  15.98280  13.298900   
10  1894523  10.16180        NaN   9.63006  13.20810   9.15029  11.491300   
11  1897841       NaN  11.606100   7.60000   8.0060